# Record language-grounded vision outputs

This notebook replays seven saved Grounding DINO tiny + SAM base runs and provides the exact recording workflow. The recording script was executed on CPU; this convenience notebook has not been executed end-to-end. No live models are needed for the lecture.

The clean authored scene has no boxes, masks or target labels. Model outputs are kept even when incorrect. Generative and reasoning interfaces in the explorer are reference examples; no Florence-2 or LISA inference is claimed.


In [ ]:
from pathlib import Path
import json, sys, subprocess, hashlib
ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents] for p in [base, base/'attention-followups/language-grounded-vision'] if (p/'src/record_models.py').exists()), None)
assert ROOT is not None, 'Open this notebook from the lecture folder or repository.'
DATA = json.loads((ROOT/'results/grounded_vision_outputs.json').read_text())
print(DATA['models'])
print(DATA['runtime'])


## Replay the recorded data first

An alignment score is not calibrated correctness probability. The SAM quality estimate is not measured mask IoU. Compare target identity before evaluating shape.


In [ ]:
for image in DATA['images']:
    assert hashlib.sha256((ROOT/image['path']).read_bytes()).hexdigest() == image['sha256']
for run in DATA['runs']:
    print(run['image_id'], repr(run['prompt']), [(d['text'], round(d['grounding_score'], 3)) for d in run['detections']])


In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display
run = next(r for r in DATA['runs'] if r['image_id']=='scene' and r['prompt']=='ball beside the dog.')
image = next(i for i in DATA['images'] if i['id']==run['image_id'])
im = Image.open(ROOT/image['path']).convert('RGB')
draw = ImageDraw.Draw(im)
for d in run['detections']:
    draw.rectangle(d['box_xyxy'], outline='#B98224', width=4)
display(im)


## Re-record (optional, downloads about 1 GB once)

Use a Python 3.12 environment with `requirements-models.txt`. Checkpoint revisions are pinned in `src/prepare_models.py`. Run the next cells only when you intend to replace the saved recordings. The clean PNG inputs are included; no browser is required for inference. Model weights are local under `output/model-cache/`.

From a terminal: `python -m pip install -r requirements-models.txt`. Do not run an installer automatically during class.


In [ ]:
subprocess.run([sys.executable, str(ROOT/'src/prepare_models.py')], check=True)


In [ ]:
subprocess.run([sys.executable, str(ROOT/'src/record_models.py')], check=True)
DATA = json.loads((ROOT/'results/grounded_vision_outputs.json').read_text())
print('Recorded queries:', len(DATA['runs']))


## Compare a new recording

The recording kit reproduces the numerical outputs. Re-running the recorder replaces `results/grounded_vision_outputs.json`; use the replay cells above to inspect it. The published HTML pages are self-contained snapshots of the original recordings and do not change when this file is replaced.
